# Projet — Enrichissement des données

Ce notebook enrichit le dataset ERS avec 4 sources externes.
Les données récupérées correspondent aux **années du dataset : 2013, 2016, 2020, 2022, 2023**.

| Source | Type | Ce qu'elle apporte |
|---|---|---|
| USDA NASS | API REST | Production agricole par état et par année |
| Open-Meteo | API REST | Météo historique par état et par année |
| EIA | API REST | Prix diesel et électricité par année |
| World Bank | Fichier | Prix des engrais par année |
| BLS | Scraping | Prix retail les plus récents disponibles |

**Fichier d'entrée** : `fruits_legumes.csv`  
**Fichier de sortie** : `fruits_legumes_enrichi.csv`

In [15]:
import pandas as pd
import numpy as np
import requests
import time
import warnings
warnings.filterwarnings('ignore')

ROUGE = '#C0392B'
BLEU  = '#3498DB'
VERT  = '#2ECC71'

# Années présentes dans notre dataset
ANNEES = [2013, 2016, 2020, 2022, 2023]

print('OK')

OK


## 1. Chargement

In [16]:
df = pd.read_csv('fruits_legumes.csv', sep=';', encoding='utf-8')

print(f'Lignes   : {df.shape[0]}')
print(f'Annees   : {sorted(df["annee"].unique())}')
df.head(3)

Lignes   : 704
Annees   : [2013, 2016, 2020, 2022, 2023]


,produit,forme,prix_detail,rendement,taille_cup,prix_cup,categorie,annee,forme_encoded,categorie_encoded
0,Watermelon,Fresh,0.3334,0.520000,0.330693,0.2120,fruit,2013,0,1
1,Turnip greens,Frozen,1.4730,0.776027,0.352740,0.6696,legume,2013,2,0
2,Turnip greens,Fresh,2.4717,0.750000,0.319670,1.0535,legume,2013,0,0


---
## SOURCE 1 — API NASS : Production par année

On récupère le volume de production par état pour chaque produit et chaque année.
Cela nous permet de savoir où est cultivé chaque produit.

In [17]:
CLE_NASS = 'A9273C62-28A4-33B3-9E51-EFBD2EA33AED'

# Correspondance nom ERS → code NASS + unité de mesure
ERS_VERS_NASS = {
    'Apples'      : ('APPLES',      'LB'),
    'Blueberries' : ('BLUEBERRIES', 'LB'),
    'Strawberries': ('STRAWBERRIES','CWT'),
    'Tomatoes'    : ('TOMATOES',    'CWT'),
    'Broccoli'    : ('BROCCOLI',    'CWT'),
    'Grapes'      : ('GRAPES',      'TONS'),
    'Peaches'     : ('PEACHES',     'TONS'),
    'Cherries'    : ('CHERRIES',    'LB'),
    'Oranges'     : ('ORANGES',     'TONS'),
    'Carrots'     : ('CARROTS',     'CWT'),
    'Lettuce'     : ('LETTUCE',     'CWT'),
    'Potatoes'    : ('POTATOES',    'CWT'),
    'Spinach'     : ('SPINACH',     'CWT'),
    'Onions'      : ('ONIONS',      'CWT'),
    'Cucumbers'   : ('CUCUMBERS',   'CWT'),
    'Cauliflower' : ('CAULIFLOWER', 'CWT'),
}

print(f'{len(ERS_VERS_NASS)} produits configurés')

16 produits configurés


In [18]:
def appeler_nass(produit_nass, unite, cle, annee):
    """Récupère la production par état pour un produit et une année."""
    url = 'https://quickstats.nass.usda.gov/api/api_GET/'
    params = {
        'key'               : cle,
        'commodity_desc'    : produit_nass,
        'statisticcat_desc' : 'PRODUCTION',
        'unit_desc'         : unite,
        'year'              : annee,
        'agg_level_desc'    : 'STATE',
        'format'            : 'JSON'
    }
    try:
        time.sleep(0.3)
        rep = requests.get(url, params=params, timeout=15)
        if rep.status_code == 200:
            return rep.json().get('data', [])
        return []
    except Exception:
        return []


print('Appel API NASS pour toutes les années...')
resultats_nass = []

for annee in ANNEES:
    print(f'  Année {annee}...')
    for nom_ers, (nom_nass, unite) in ERS_VERS_NASS.items():
        donnees = appeler_nass(nom_nass, unite, CLE_NASS, annee)
        for ligne in donnees:
            if ligne.get('state_name') == 'OTHER STATES':
                continue
            resultats_nass.append({
                'produit'       : nom_ers,
                'annee'         : annee,
                'etat'          : ligne.get('state_name'),
                'code_etat'     : ligne.get('state_alpha'),
                'production_raw': ligne.get('Value'),
                'unite'         : unite,
            })

df_nass_raw = pd.DataFrame(resultats_nass)
print(f'\nNASS brut : {df_nass_raw.shape[0]} lignes')

Appel API NASS pour toutes les années...
  Année 2013...
  Année 2016...
  Année 2020...
  Année 2022...
  Année 2023...

NASS brut : 3952 lignes


In [19]:
# Conversion de toutes les unités en livres pour harmoniser
CONVERSION = {'LB': 1, 'CWT': 100, 'TONS': 2000}

df_nass_raw['production'] = pd.to_numeric(
    df_nass_raw['production_raw'].astype(str).str.replace(',', ''),
    errors='coerce'
)
df_nass_raw['production_lbs'] = df_nass_raw.apply(
    lambda r: r['production'] * CONVERSION.get(r['unite'], 1), axis=1
)
df_nass_raw = df_nass_raw[
    df_nass_raw['production_lbs'] > 0
].dropna(subset=['production_lbs'])

# On garde l'état avec le plus gros volume par produit + année
idx = df_nass_raw.groupby(['produit', 'annee'])['production_lbs'].idxmax()
df_nass = df_nass_raw.loc[idx][
    ['produit', 'annee', 'etat', 'code_etat', 'production_lbs']
].copy()
df_nass.columns = ['produit', 'annee', 'etat_production',
                   'code_etat', 'production_lbs']

print(f'NASS nettoyé : {df_nass.shape[0]} lignes')
print(df_nass.head(5).to_string())

NASS nettoyé : 80 lignes
     produit  annee etat_production code_etat  production_lbs
26    Apples   2013      WASHINGTON        WA    5.900000e+09
971   Apples   2016      WASHINGTON        WA    7.320000e+09
2444  Apples   2020      WASHINGTON        WA    7.400000e+09
2951  Apples   2022      WASHINGTON        WA    6.500000e+09
3457  Apples   2023      WASHINGTON        WA    7.655000e+09


In [20]:
# Jointure sur produit + année
df['produit_cle'] = df['produit'].str.split(',').str[0].str.strip()

df = df.merge(
    df_nass,
    left_on=['produit_cle', 'annee'],
    right_on=['produit', 'annee'],
    how='left'
)
df.drop(columns=['produit_y', 'produit_cle'], inplace=True, errors='ignore')
df.rename(columns={'produit_x': 'produit'}, inplace=True)

pct = df['etat_production'].notna().mean() * 100
print(f'Correspondance NASS : {pct:.1f}%')
print(f'Dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes')

Correspondance NASS : 26.1%
Dataset : 704 lignes x 13 colonnes


---
## SOURCE 2 — API Open-Meteo : Météo par année

Pour chaque état producteur et chaque année, on récupère
température moyenne, jours de gel, jours de chaleur et précipitations.

In [21]:
GPS_ETATS = {
    'CA': (36.7783, -119.4179), 'WA': (47.3826, -120.4472),
    'FL': (27.9944,  -81.7603), 'MI': (43.3504,  -84.5603),
    'OR': (44.5720, -122.0709), 'GA': (32.1574,  -82.9071),
    'NC': (35.6301,  -79.8064), 'ID': (44.0682, -114.7420),
    'IA': (42.0046,  -93.2140), 'WI': (44.2685,  -89.6165),
    'NY': (42.1657,  -74.9481), 'PA': (40.5908,  -77.2098),
    'MT': (46.8797, -110.3626), 'HI': (20.7967, -156.3319),
}

df_nass['latitude']  = df_nass['code_etat'].map(
    lambda x: GPS_ETATS.get(x, (None, None))[0])
df_nass['longitude'] = df_nass['code_etat'].map(
    lambda x: GPS_ETATS.get(x, (None, None))[1])

print('GPS ajoutés')

GPS ajoutés


In [22]:
def appeler_meteo(lat, lon, annee):
    """Récupère 4 indicateurs météo annuels pour des coordonnées GPS."""
    url = 'https://archive-api.open-meteo.com/v1/archive'
    params = {
        'latitude'  : lat,
        'longitude' : lon,
        'start_date': f'{annee}-01-01',
        'end_date'  : f'{annee}-12-31',
        'daily'     : 'temperature_2m_max,temperature_2m_min,precipitation_sum',
        'timezone'  : 'America/New_York'
    }
    try:
        time.sleep(0.5)
        rep = requests.get(url, params=params, timeout=20)
        if rep.status_code == 200:
            d    = rep.json().get('daily', {})
            tmax = d.get('temperature_2m_max', [])
            tmin = d.get('temperature_2m_min', [])
            prec = d.get('precipitation_sum', [])
            return {
                'temp_moyenne' : round(np.nanmean(
                    [(a+b)/2 for a,b in zip(tmax,tmin)]), 2),
                'jours_gel'    : sum(1 for t in tmin if t is not None and t < 0),
                'jours_chaleur': sum(1 for t in tmax if t is not None and t > 35),
                'precip_totale': round(sum(p for p in prec if p is not None), 1),
            }
        return None
    except Exception:
        return None


print('Appel Open-Meteo pour toutes les années...')
resultats_meteo = []

# On évite les doublons : une seule requête par état + année
vus = set()
for _, row in df_nass.iterrows():
    if pd.isna(row['latitude']):
        continue
    cle = (row['code_etat'], row['annee'])
    if cle in vus:
        continue
    vus.add(cle)
    meteo = appeler_meteo(row['latitude'], row['longitude'], row['annee'])
    if meteo:
        meteo['code_etat'] = row['code_etat']
        meteo['annee']     = row['annee']
        resultats_meteo.append(meteo)
        print(f'  {row["code_etat"]} {int(row["annee"])} OK')

df_meteo = pd.DataFrame(resultats_meteo)
print(f'\nMétéo : {df_meteo.shape[0]} combinaisons état/année')

Appel Open-Meteo pour toutes les années...
  WA 2013 OK
  WA 2016 OK
  WA 2020 OK
  WA 2022 OK
  WA 2023 OK
  MI 2013 OK
  CA 2013 OK
  CA 2016 OK
  CA 2020 OK
  CA 2022 OK
  CA 2023 OK
  MI 2016 OK
  MI 2020 OK
  MI 2022 OK
  MI 2023 OK
  FL 2013 OK
  FL 2023 OK
  OR 2013 OK
  FL 2016 OK
  FL 2020 OK
  FL 2022 OK
  ID 2013 OK
  ID 2016 OK
  ID 2020 OK
  ID 2022 OK
  ID 2023 OK

Météo : 26 combinaisons état/année


In [23]:
# Jointure météo sur code_etat + annee
df = df.merge(df_meteo, on=['code_etat', 'annee'], how='left')

pct = df['temp_moyenne'].notna().mean() * 100
print(f'Correspondance météo : {pct:.1f}%')
print(f'Dataset : {df.shape[0]} lignes x {df.shape[1]} colonnes')

Correspondance météo : 26.1%
Dataset : 704 lignes x 17 colonnes


---
## SOURCE 3 — API EIA : Prix de l'énergie par année

In [24]:
CLE_EIA = 'bu79Mxq083aYl3CCTZwRjJoVT8GlUGVWx3opxKhu'

# Mapping état → région PADD (l'EIA publie par région, pas par état)
ETAT_VERS_PADD = {
    'CA': 'SCA', 'WA': 'R50', 'OR': 'R50', 'HI': 'R50',
    'ID': 'R40', 'MT': 'R40',
    'FL': 'R1Z', 'GA': 'R1Z', 'NC': 'R1Z',
    'PA': 'R1Y', 'NY': 'R1Y',
    'MI': 'R20', 'IA': 'R20', 'WI': 'R20',
}

def appeler_eia_diesel(padd, cle, annee):
    """Prix moyen du diesel pour une région PADD et une année."""
    url = 'https://api.eia.gov/v2/petroleum/pri/gnd/data/'
    params = {
        'api_key'           : cle,
        'frequency'         : 'monthly',
        'data[0]'           : 'value',
        'facets[duoarea][]' : padd,
        'facets[product][]' : 'EPD2D',
        'facets[process][]' : 'PTE',
        'start'             : f'{annee}-01',
        'end'               : f'{annee}-12',
    }
    try:
        time.sleep(0.3)
        rep = requests.get(url, params=params, timeout=15)
        if rep.status_code == 200:
            donnees = rep.json().get('response', {}).get('data', [])
            valeurs = [float(d['value']) for d in donnees
                       if d.get('value') not in [None, 'null', '']]
            return round(np.mean(valeurs), 4) if valeurs else None
        return None
    except Exception:
        return None

def appeler_eia_electricite(code_etat, cle, annee):
    """Prix moyen de l'électricité pour un état et une année."""
    url = 'https://api.eia.gov/v2/electricity/retail-sales/data/'
    params = {
        'api_key'            : cle,
        'frequency'          : 'annual',
        'data[0]'            : 'price',
        'facets[stateid][]'  : code_etat,
        'facets[sectorid][]' : 'ALL',
        'start'              : str(annee),
        'end'                : str(annee),
    }
    try:
        time.sleep(0.3)
        rep = requests.get(url, params=params, timeout=15)
        if rep.status_code == 200:
            donnees = rep.json().get('response', {}).get('data', [])
            valeurs = [float(d['price']) for d in donnees
                       if d.get('price') not in [None, 'null', '']]
            return round(np.mean(valeurs), 4) if valeurs else None
        return None
    except Exception:
        return None


print('Appel API EIA pour toutes les années...')
resultats_eia = []

# Combinaisons uniques état + année présentes dans le dataset
for code, annee in df[['code_etat', 'annee']].dropna().drop_duplicates().values:
    padd   = ETAT_VERS_PADD.get(code, 'NUS')
    diesel = appeler_eia_diesel(padd, CLE_EIA, int(annee))
    elec   = appeler_eia_electricite(code, CLE_EIA, int(annee))
    resultats_eia.append({
        'code_etat'       : code,
        'annee'           : int(annee),
        'prix_diesel'     : diesel,
        'prix_electricite': elec,
    })
    print(f'  {code} {int(annee)} — diesel={diesel}  elec={elec}')

df_eia = pd.DataFrame(resultats_eia)

# Jointure sur code_etat + annee
df = df.merge(df_eia, on=['code_etat', 'annee'], how='left')

# NaN remplacés par la médiane nationale
for col in ['prix_diesel', 'prix_electricite']:
    df[col].fillna(df[col].median(), inplace=True)

print(f'\nDataset : {df.shape[0]} lignes x {df.shape[1]} colonnes')

Appel API EIA pour toutes les années...
  CA 2013 — diesel=4.1267  elec=14.3
  ID 2013 — diesel=3.8763  elec=7.57
  WA 2013 — diesel=4.0522  elec=7.09
  MI 2013 — diesel=3.9042  elec=11.21
  OR 2013 — diesel=4.0522  elec=8.44
  FL 2013 — diesel=3.8773  elec=10.22
  CA 2016 — diesel=2.6558  elec=15.23
  MI 2016 — diesel=2.2612  elec=11.05
  ID 2016 — diesel=2.3115  elec=8.08
  WA 2016 — diesel=2.5602  elec=7.68
  FL 2016 — diesel=2.2464  elec=9.91
  CA 2020 — diesel=3.3808  elec=18.0
  MI 2020 — diesel=2.4306  elec=12.21
  WA 2020 — diesel=3.0878  elec=8.33
  FL 2020 — diesel=2.4849  elec=10.06
  ID 2020 — diesel=2.5207  elec=7.99
  CA 2022 — diesel=6.0362  elec=22.33
  ID 2022 — diesel=4.9854  elec=8.51
  MI 2022 — diesel=4.9072  elec=13.2
  FL 2022 — diesel=4.9313  elec=12.51
  WA 2022 — diesel=5.6248  elec=9.05
  WA 2023 — diesel=4.9428  elec=9.58
  CA 2023 — diesel=5.3592  elec=24.87
  MI 2023 — diesel=4.1142  elec=13.68
  ID 2023 — diesel=4.3575  elec=9.08

Dataset : 704 lignes x 1

---
## SOURCE 4 — World Bank : Prix des engrais par année

Source : World Bank Pink Sheet Mars 2026.
Les prix sont mondiaux — identiques pour tous les produits d'une même année.

In [25]:
# Prix des engrais par année ($/mt) — World Bank Pink Sheet
# urea = urée, dap = phosphate, tsp = triple superphosphate, mop = potasse
ENGRAIS_PAR_ANNEE = {
    2013: {'urea': 304.0, 'dap': 474.0, 'tsp': 390.0, 'mop': 355.0},
    2016: {'urea': 212.0, 'dap': 330.0, 'tsp': 270.0, 'mop': 215.0},
    2020: {'urea': 233.0, 'dap': 344.0, 'tsp': 286.0, 'mop': 213.0},
    2022: {'urea': 726.0, 'dap': 901.0, 'tsp': 735.0, 'mop': 487.0},
    2023: {'urea': 358.0, 'dap': 550.0, 'tsp': 480.0, 'mop': 383.0},
}

df_engrais = pd.DataFrame([
    {'annee': annee, **vals}
    for annee, vals in ENGRAIS_PAR_ANNEE.items()
])

# Jointure sur l'année uniquement
df = df.merge(df_engrais, on='annee', how='left')

print('Engrais ajoutés :')
print(df_engrais)
print(f'\nDataset : {df.shape[0]} lignes x {df.shape[1]} colonnes')

Engrais ajoutés :
   annee   urea    dap    tsp    mop
0   2013  304.0  474.0  390.0  355.0
1   2016  212.0  330.0  270.0  215.0
2   2020  233.0  344.0  286.0  213.0
3   2022  726.0  901.0  735.0  487.0
4   2023  358.0  550.0  480.0  383.0

Dataset : 704 lignes x 23 colonnes


---
## SOURCE 5 — Scraping BLS : Prix retail

Le BLS publie uniquement les prix les plus récents (pas d'historique).
On utilise ces prix comme référence comparative.

In [26]:
URL_BLS = 'https://www.bls.gov/regions/mid-atlantic/data/averageretailfoodandenergyprices_usandwest_table.htm'
headers = {'User-Agent': 'Mozilla/5.0 (compatible; OrientIA-Bot/1.0)'}

print('Scraping BLS...')
df_bls = pd.DataFrame(columns=['produit_cle', 'prix_bls'])

try:
    time.sleep(1)
    rep    = requests.get(URL_BLS, headers=headers, timeout=20)
    tables = pd.read_html(rep.text)
    table  = tables[0].copy()

    # Aplatissement des colonnes multi-niveaux
    table.columns = ['_'.join([str(c) for c in col
                     if 'Unnamed' not in str(c)]).strip()
                     for col in table.columns]

    df_bls_raw = pd.DataFrame({
        'produit_bls': table.iloc[2:, 0].values,
        'prix_bls'   : pd.to_numeric(table.iloc[2:, 2].values, errors='coerce'),
    }).dropna(subset=['prix_bls'])

    # On garde uniquement les fruits et légumes
    mots   = ['apple','banana','orange','grape','peach','strawberr',
              'potato','lettuce','tomato','broccoli','carrot','onion']
    masque = df_bls_raw['produit_bls'].astype(str).str.lower()\
               .str.contains('|'.join(mots), na=False)
    df_bls = df_bls_raw[masque].copy()
    df_bls['produit_cle'] = df_bls['produit_bls'].str.split(',').str[0]\
                              .str.lower().str.strip()

    print(f'BLS : {len(df_bls)} produits scrapés')
    print(df_bls[['produit_bls', 'prix_bls']].to_string())

except Exception as e:
    print(f'Scraping échoué : {e}')

Scraping BLS...
BLS : 10 produits scrapés
                                                            produit_bls  prix_bls
54                                          Bananas, per lb. (453.6 gm)     0.619
55                                   Oranges, Navel, per lb. (453.6 gm)     1.536
58                                       Grapefruit, per lb. (453.6 gm)     1.668
63                        Strawberries, dry pint, per 12 oz. (340.2 gm)     2.905
64                                  Potatoes, white, per lb. (453.6 gm)     0.957
65                                 Lettuce, iceberg, per lb. (453.6 gm)     1.522
66                                 Lettuce, romaine, per lb. (453.6 gm)     2.865
67                            Tomatoes, field grown, per lb. (453.6 gm)     1.848
74  Orange juice, frozen concentrate, 12 oz. can, per 16 oz. (473.2 ml)     4.492
89                                             Potato chips, per 16 oz.     6.462


In [27]:
# Jointure BLS sur le nom du produit
df['produit_cle'] = df['produit'].str.split(',').str[0].str.lower().str.strip()

df = df.merge(
    df_bls[['produit_cle', 'prix_bls']],
    on='produit_cle',
    how='left'
)
df['prix_bls'].fillna(df['prix_bls'].median(), inplace=True)
df.drop(columns=['produit_cle'], inplace=True, errors='ignore')

print(f'Dataset final : {df.shape[0]} lignes x {df.shape[1]} colonnes')

Dataset final : 710 lignes x 24 colonnes


### Vérification du dataset et nettoyage ###

In [31]:
# Analyse complète des valeurs manquantes
print('=' * 60)
print('VALEURS MANQUANTES PAR COLONNE')
print('=' * 60)
print(df.isnull().sum())

print('\n' + '=' * 60)
print('VALEURS MANQUANTES PAR ANNEE ET COLONNE')
print('=' * 60)
cols_nan = [c for c in df.columns if df[c].isnull().any()]
for col in cols_nan:
    print(f'\n{col} :')
    print(df.groupby('annee')[col].apply(
        lambda x: f'  {x.isnull().sum()} NaN sur {len(x)} lignes'
    ))

print('\n' + '=' * 60)
print('PRODUITS SANS ETAT PRODUCTEUR')
print('=' * 60)
sans_etat = df[df['etat_production'].isna()]['produit'].unique()
print(f'{len(sans_etat)} produits sans état :')
for p in sorted(sans_etat)[:30]:
    print(f'  {p}')

VALEURS MANQUANTES PAR COLONNE
produit                0
forme                  0
prix_detail            0
rendement              0
taille_cup             0
prix_cup               0
categorie              0
annee                  0
forme_encoded          0
categorie_encoded      0
etat_production      520
code_etat            520
production_lbs       520
temp_moyenne         520
jours_gel            520
jours_chaleur        520
precip_totale        520
prix_diesel            0
prix_electricite       0
urea                   0
dap                    0
tsp                    0
mop                    0
prix_bls               0
dtype: int64

VALEURS MANQUANTES PAR ANNEE ET COLONNE

etat_production :
annee
2013       95 NaN sur 125 lignes
2016       93 NaN sur 123 lignes
2020       91 NaN sur 121 lignes
2022      128 NaN sur 183 lignes
2023      113 NaN sur 158 lignes
Name: etat_production, dtype: object

code_etat :
annee
2013       95 NaN sur 125 lignes
2016       93 NaN sur 123 lignes
202

In [32]:
# Stratégie de nettoyage des NaN

# Les valeurs manquantes concernent principalement les produits importés non recensés par le NASS.# 
#Pour ne pas perdre ces lignes, on les remplace par la médiane de leur catégorie et de leur année.#
#On choisit la médiane car elle est robuste aux valeurs extrêmes — la production californienne ne fausse pas le résultat.#
#C'est une imputation conservatrice qui ne crée pas d'information artificielle.#

print('Remplacement des NaN...')

# production_lbs — médiane par année
df['production_lbs'] = df.groupby('annee')['production_lbs']\
                         .transform(lambda x: x.fillna(x.median()))

# Météo — médiane par année ET catégorie
# Justification : un fruit tropical a un profil météo différent d'un légume tempéré
for col in ['temp_moyenne', 'jours_gel', 'jours_chaleur', 'precip_totale']:
    df[col] = df.groupby(['annee', 'categorie'])[col]\
                .transform(lambda x: x.fillna(x.median()))

# Vérification
print('\nNaN restants après remplacement :')
cols = ['production_lbs', 'temp_moyenne', 'jours_gel',
        'jours_chaleur', 'precip_totale']
for col in cols:
    nb = df[col].isnull().sum()
    print(f'  {col:<20} : {nb} NaN')

print('\nAperçu après remplacement :')
print(df[['produit', 'annee', 'categorie',
          'temp_moyenne', 'jours_gel', 'production_lbs']].head(8))

Remplacement des NaN...

NaN restants après remplacement :
  production_lbs       : 0 NaN
  temp_moyenne         : 0 NaN
  jours_gel            : 0 NaN
  jours_chaleur        : 0 NaN
  precip_totale        : 0 NaN

Aperçu après remplacement :
          produit  annee categorie  temp_moyenne  jours_gel  production_lbs
0      Watermelon   2013     fruit         18.55        6.0    2.401650e+09
1   Turnip greens   2013    legume         18.55        6.0    2.401650e+09
2   Turnip greens   2013    legume         18.55        6.0    2.401650e+09
3   Turnip greens   2013    legume         18.55        6.0    2.401650e+09
4        Tomatoes   2013    legume         18.55        6.0    1.020000e+09
5        Potatoes   2013    legume          1.20      252.0    1.329250e+10
6         Pumpkin   2013    legume         18.55        6.0    2.401650e+09
7  Mustard greens   2013    legume         18.55        6.0    2.401650e+09


In [33]:
# Vérification finale avant export
print('Valeurs manquantes restantes :')
print(df.isnull().sum()[df.isnull().sum() > 0])

print(f'\nDimensions : {df.shape[0]} lignes x {df.shape[1]} colonnes')
print(f'Années     : {sorted(df["annee"].unique())}')
print(f'\nAperçu :')
print(df[['produit', 'annee', 'categorie', 'prix_cup',
          'temp_moyenne', 'jours_gel', 'prix_diesel', 'urea']].head(5))

Valeurs manquantes restantes :
etat_production    520
code_etat          520
dtype: int64

Dimensions : 710 lignes x 24 colonnes
Années     : [2013, 2016, 2020, 2022, 2023]

Aperçu :
         produit  annee categorie  prix_cup  temp_moyenne  jours_gel  \
0     Watermelon   2013     fruit    0.2120         18.55        6.0   
1  Turnip greens   2013    legume    0.6696         18.55        6.0   
2  Turnip greens   2013    legume    1.0535         18.55        6.0   
3  Turnip greens   2013    legume    0.5238         18.55        6.0   
4       Tomatoes   2013    legume    0.4995         18.55        6.0   

   prix_diesel   urea  
0       4.2421  304.0  
1       4.2421  304.0  
2       4.2421  304.0  
3       4.2421  304.0  
4       4.1267  304.0  


etat_production et code_etat sont des colonnes texte qu'on ne peut pas imputer — c'est normal, les produits importés n'ont pas d'état américain. Ces colonnes ne seront pas utilisées dans le modèle ML.

## 6. Export

In [34]:
df.to_csv('fruits_legumes_enrichi.csv', index=False, sep=';', encoding='utf-8')

print('=' * 55)
print('EXPORT OK')
print('=' * 55)
print(f'Fichier  : fruits_legumes_enrichi.csv')
print(f'Lignes   : {df.shape[0]}')
print(f'Colonnes : {df.shape[1]}')
print(f'Annees   : {sorted(df["annee"].unique())}')
print()
print('Colonnes ajoutées :')
print('  NASS       — etat_production, code_etat, production_lbs (par année)')
print('  Open-Meteo — temp_moyenne, jours_gel, jours_chaleur, precip_totale (par année)')
print('  EIA        — prix_diesel, prix_electricite (par année)')
print('  World Bank — urea, dap, tsp, mop (par année)')
print('  BLS        — prix_bls (référence retail)')
df.head(3)

EXPORT OK
Fichier  : fruits_legumes_enrichi.csv
Lignes   : 710
Colonnes : 24
Annees   : [2013, 2016, 2020, 2022, 2023]

Colonnes ajoutées :
  NASS       — etat_production, code_etat, production_lbs (par année)
  Open-Meteo — temp_moyenne, jours_gel, jours_chaleur, precip_totale (par année)
  EIA        — prix_diesel, prix_electricite (par année)
  World Bank — urea, dap, tsp, mop (par année)
  BLS        — prix_bls (référence retail)


,produit,forme,prix_detail,rendement,taille_cup,prix_cup,categorie,annee,forme_encoded,categorie_encoded,...,jours_gel,jours_chaleur,precip_totale,prix_diesel,prix_electricite,urea,dap,tsp,mop,prix_bls
0,Watermelon,Fresh,0.3334,0.520000,0.330693,0.2120,fruit,2013,0,1,...,6.0,1.0,511.2,4.2421,14.3,304.0,474.0,390.0,355.0,1.536
1,Turnip greens,Frozen,1.4730,0.776027,0.352740,0.6696,legume,2013,2,0,...,6.0,74.0,142.1,4.2421,14.3,304.0,474.0,390.0,355.0,1.536
2,Turnip greens,Fresh,2.4717,0.750000,0.319670,1.0535,legume,2013,0,0,...,6.0,74.0,142.1,4.2421,14.3,304.0,474.0,390.0,355.0,1.536
